# Aura Knowledge Distillation (Kaggle - T4 GPU)
Trains our custom UniversalDenseCore model to mimic GPT-2.

**Setup:**
1. Kaggle → Create → New Notebook
2. **Settings → Accelerator → GPU T4**
3. **Settings → Internet → On**
4. Run All

In [ ]:
!pip install transformers datasets torch tqdm tokenizers -q
print('OK')

In [ ]:
# Clone the repo to get our model code
import os, subprocess
if not os.path.exists('Aura-v1'):
    !git clone https://github.com/mithunsuresh121-stack/Aura-v1.git
# Copy our files to working directory
!cp Aura-v1/cortex-train/model.py .
!cp Aura-v1/cortex-train/hypernetwork.py .
!cp Aura-v1/cortex-train/distill.py .
print('Files ready: model.py, hypernetwork.py, distill.py')

In [ ]:
# Test imports
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}')
from model import UniversalDenseCore
print('Model import OK')

In [ ]:
# Run distillation
# Config: 50M param student (d_model=512, n_layers=8, n_heads=8, d_ff=2048)
# ~8 hours on T4 for 30K steps (100K would exceed Kaggle time limit)
!python distill.py \
    --steps 30000 \
    --batch-size 8 \
    --seq-len 256 \
    --lr 1e-4 \
    --temp 4.0 \
    --save-every 5000 \
    --checkpoint-dir distilled_checkpoints \
    --d-model 512 \
    --n-layers 8 \
    --n-heads 8 \
    --d-ff 2048
print('DISTILLATION DONE')

In [ ]:
# List checkpoints
import os
for f in sorted(os.listdir('distilled_checkpoints')):
    path = os.path.join('distilled_checkpoints', f)
    print(f'{f}  ({os.path.getsize(path)/1e6:.1f} MB)')

In [ ]:
# Quick test generation
from distill import get_gpt2_tokenizer
from model import UniversalDenseCore
import torch

ckpt = torch.load('distilled_checkpoints/distilled_final.pt', map_location='cuda')
config = ckpt['config']
model = UniversalDenseCore(**config).to('cuda')
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded: {sum(p.numel() for p in model.parameters()):,} params')

tokenizer = get_gpt2_tokenizer()
prompt = 'Once upon a time'
prompt_ids = tokenizer.encode(prompt, return_tensors='pt').to('cuda')

output, _ = model.generate(
    prompt_ids, max_new_tokens=64, temperature=0.8, top_k=40,
    repetition_penalty=1.1,
)
print(f'Prompt: {prompt}')
print(f'Output: {tokenizer.decode(output[0].tolist())}')

In [ ]:
# Download instructions
print('To download:')
print('1. Click on "distilled_checkpoints/" in the file browser (right side)')
print('2. Select distilled_final.pt')
print('3. Click Download')
print('')
print('Then on your Mac:')
print('  cp distilled_final.pt cortex-train/checkpoints/')
print('  The server will automatically use it')